In [7]:
import pandas as pd
import os

# 1. 파일 경로
menu_file_path = "./식당대12중53소132상세메뉴379분류.csv"
search_files_paths = [
    "./상세메뉴네이버_검색_절대숫자.xlsx",
    "./소분류상세메뉴네이버_검색_절대숫자.xlsx",
    "./음식상세메뉴네이버_검색_절대숫자.xlsx"
]

# 2. 메뉴 파일 불러오기
menu_df = pd.read_csv(menu_file_path)

# 3. 상세메뉴 분해 (순서 보존을 위해 index 유지)
menu_expanded = []
menu_count = 0
for idx, row in menu_df.iterrows():
    detail_list = [item.strip() for item in row['상세메뉴'].split(',')]
    for detail in detail_list:
        menu_expanded.append({
            '원래순서': menu_count,
            '대분류': row['대분류'],
            '중분류': row['중분류'],
            '소분류': row['소분류'],
            '상세메뉴': detail
        })
        menu_count+=1

expanded_menu_df = pd.DataFrame(menu_expanded)

# 4. 검색 파일 불러와서 병합
search_dfs = []
for path in search_files_paths:
    xls = pd.ExcelFile(path)
    for sheet in xls.sheet_names:
        df = xls.parse(sheet)
        df['출처파일'] = os.path.basename(path)
        df['출처시트'] = sheet
        search_dfs.append(df)

combined_search_df = pd.concat(search_dfs, ignore_index=True)

# 5. 정확히 일치하는 키워드 기준 병합
search_cols = ['블로그_검색수', '뉴스_검색수', '카페_검색수', '웹_검색수', '총합']
merged_df = expanded_menu_df.merge(
    combined_search_df[['원본키워드'] + search_cols],
    left_on='상세메뉴',
    right_on='원본키워드',
    how='left'
)

# 6. 상세메뉴별 검색수 평균 계산
grouped_avg_df = merged_df.groupby(
    ['원래순서', '대분류', '중분류', '소분류', '상세메뉴'],
    as_index=False
)[search_cols].mean()
print(grouped_avg_df)
# 7. 원래 순서대로 정렬
grouped_avg_df = grouped_avg_df.sort_values(by='원래순서').drop(columns=['원래순서'])

# 8. 결과 저장
output_path = "./키워드_검색수_평균_결과_순서유지.xlsx"
grouped_avg_df.to_excel(output_path, index=False)

     원래순서  대분류     중분류    소분류     상세메뉴       블로그_검색수        뉴스_검색수  \
0       0   한식  백반/가정식   제육볶음     제육볶음  1.428292e+06  24102.333333   
1       1   한식  백반/가정식   제육볶음   매운제육볶음  1.701000e+05   1737.666667   
2       2   한식  백반/가정식   제육볶음   두부제육볶음  2.325257e+05   1986.000000   
3       3   한식  백반/가정식    찌개류     된장찌개  2.693716e+06  31999.000000   
4       4   한식  백반/가정식    찌개류     김치찌개  1.711412e+06  46609.333333   
..    ...  ...     ...    ...      ...           ...           ...   
376   376  건강식      비건  비건디저트  두유아이스크림  6.349300e+04   1622.333333   
377   377  건강식     헬스식   저칼로리  닭가슴살샐러드  5.812350e+05   9384.666667   
378   378  건강식     헬스식   저칼로리     퀴노아볼  2.564567e+04   1211.000000   
379   379  건강식     헬스식    단백질     프로틴볼  1.066090e+05   6074.000000   
380   380  건강식     헬스식    단백질    그릭요거트  4.958987e+05   8585.333333   

            카페_검색수         웹_검색수            총합  
0    284528.666667  2.311095e+06  4.048019e+06  
1     20487.666667  6.308837e+05  8.232090e+05  
2     46769.